# Zomato Análise Espacial
 //zomato spacial analysis
 

In [5]:
#importando bibliotecas para análise espacial
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import plotly as plot
import seaborn as sns
from geopy.geocoders import Nominatim
import folium
from folium.plugins import HeatMap
from folium.plugins import FastMarkerCluster
import kagglehub

In [6]:
#importando arquivos
#importing files

#urls
urls = ['https://raw.githubusercontent.com/CarolineMNves/Zomato_an-lise_espacial/refs/heads/main/files/zomato_part0.csv',
        'https://raw.githubusercontent.com/CarolineMNves/Zomato_an-lise_espacial/refs/heads/main/files/zomato_part1.csv',
        'https://raw.githubusercontent.com/CarolineMNves/Zomato_an-lise_espacial/refs/heads/main/files/zomato_part10.csv',
        'https://raw.githubusercontent.com/CarolineMNves/Zomato_an-lise_espacial/refs/heads/main/files/zomato_part2.csv',
        'https://raw.githubusercontent.com/CarolineMNves/Zomato_an-lise_espacial/refs/heads/main/files/zomato_part3.csv',
        'https://raw.githubusercontent.com/CarolineMNves/Zomato_an-lise_espacial/refs/heads/main/files/zomato_part4.csv',
        'https://raw.githubusercontent.com/CarolineMNves/Zomato_an-lise_espacial/refs/heads/main/files/zomato_part5.csv',
        'https://raw.githubusercontent.com/CarolineMNves/Zomato_an-lise_espacial/refs/heads/main/files/zomato_part6.csv',
        'https://raw.githubusercontent.com/CarolineMNves/Zomato_an-lise_espacial/refs/heads/main/files/zomato_part7.csv',
        'https://raw.githubusercontent.com/CarolineMNves/Zomato_an-lise_espacial/refs/heads/main/files/zomato_part8.csv',
        'https://raw.githubusercontent.com/CarolineMNves/Zomato_an-lise_espacial/refs/heads/main/files/zomato_part9.csv']

#reading // lendo
df_zomato = pd.concat([pd.read_csv(url) for url in urls])
print(type(df_zomato))
print('==*=='*27)
print(df_zomato.head(3))
print('==*=='*27)
print(df_zomato.tail())

<class 'pandas.core.frame.DataFrame'>
==*====*====*====*====*====*====*====*====*====*====*====*====*====*====*====*====*====*====*====*====*====*====*====*====*====*====*==
                                                 url  \
0  https://www.zomato.com/bangalore/jalsa-banasha...   
1  https://www.zomato.com/bangalore/spice-elephan...   
2  https://www.zomato.com/SanchurroBangalore?cont...   

                                             address             name  \
0  942, 21st Main Road, 2nd Stage, Banashankari, ...            Jalsa   
1  2nd Floor, 80 Feet Road, Near Big Bazaar, 6th ...   Spice Elephant   
2  1112, Next to KIMS Medical College, 17th Cross...  San Churro Cafe   

  online_order book_table   rate  votes                           phone  \
0          Yes        Yes  4.1/5    775  080 42297555\r\n+91 9743772233   
1          Yes         No  4.1/5    787                    080 41714161   
2          Yes         No  3.8/5    918                  +91 9663487993   

       

In [7]:
print(df_zomato.info())
print('==*=='*27)
print(df_zomato.duplicated().sum())
print('==*=='*27)
print(df_zomato.isnull().sum())

<class 'pandas.core.frame.DataFrame'>
Index: 51717 entries, 0 to 4999
Data columns (total 17 columns):
 #   Column                       Non-Null Count  Dtype 
---  ------                       --------------  ----- 
 0   url                          51717 non-null  object
 1   address                      51717 non-null  object
 2   name                         51717 non-null  object
 3   online_order                 51717 non-null  object
 4   book_table                   51717 non-null  object
 5   rate                         43942 non-null  object
 6   votes                        51717 non-null  int64 
 7   phone                        50509 non-null  object
 8   location                     51696 non-null  object
 9   rest_type                    51490 non-null  object
 10  dish_liked                   23639 non-null  object
 11  cuisines                     51672 non-null  object
 12  approx_cost(for two people)  51371 non-null  object
 13  reviews_list                 51717 no

Como o objetivo será a análise e, posteriormente, implementação de técnica de ML, todas as linhas com entradas nulas para
as colunas 'rate', 'location' e 'rest_type' que possuem informações de nota, localização e tipo de restaurante, respectivamente, serão removidas.

In [8]:
df_zomato.dropna(subset= ['rate','location','rest_type'], inplace= True)
print(df_zomato.isna().sum())

url                                0
address                            0
name                               0
online_order                       0
book_table                         0
rate                               0
votes                              0
phone                            828
location                           0
rest_type                          0
dish_liked                     20252
cuisines                          11
approx_cost(for two people)      250
reviews_list                       0
menu_item                          0
listed_in(type)                    0
listed_in(city)                    0
dtype: int64


#### Tratamento

In [9]:
zomato2 = df_zomato.copy()
zomato2['location'] = zomato2['location'] + ', Bangalore, Karnataka, India'
zomato2['location']

0       Banashankari, Bangalore, Karnataka, India
1       Banashankari, Bangalore, Karnataka, India
2       Banashankari, Bangalore, Karnataka, India
3       Banashankari, Bangalore, Karnataka, India
4       Basavanagudi, Bangalore, Karnataka, India
                          ...                    
4995       Bellandur, Bangalore, Karnataka, India
4996       Bellandur, Bangalore, Karnataka, India
4997       Bellandur, Bangalore, Karnataka, India
4998       Bellandur, Bangalore, Karnataka, India
4999       Bellandur, Bangalore, Karnataka, India
Name: location, Length: 43791, dtype: object

extraindo coordenadas de lat e long de atributos com nome e linhas de endereço

In [10]:
lat_long = pd.DataFrame()
lat_long['Name'] = zomato2['location'].unique()
lat_long.head()

,Name
0,"Banashankari, Bangalore, Karnataka, India"
1,"Basavanagudi, Bangalore, Karnataka, India"
2,"Mysore Road, Bangalore, Karnataka, India"
3,"Jayanagar, Bangalore, Karnataka, India"
4,"Kumaraswamy Layout, Bangalore, Karnataka, India"


In [11]:
geolocator = Nominatim(user_agent= 'app', timeout= None)

lat = []
lon= []

for name in lat_long['Name']:
  location= geolocator.geocode(name)

  if location is None:
    lat.append(np.nan)
    lon.append(np.nan)
  else:
    lat.append(location.latitude)
    lon.append(location.longitude)

print(f'Latidudes encontradas: {lat}')
print('=*='*22)
print(f'Longitudes encontradas: {lon}')

Latidudes encontradas: [12.9393328, 12.9417261, 12.953397861216981, 12.9399039, 12.9067683, 12.9274413, 12.9660722, 12.9055682, 12.9096941, 12.864107149999999, 12.965717999999999, 12.951855665820787, 12.9163603, 12.93909799796322, 12.9089453, 12.9854892, 12.848759900000001, 12.9489339, 12.9575547, 12.9348429, 12.970018575089048, 12.90056335, 12.9552572, 12.9364846, 12.92535245, 12.924013794074499, 12.9696365, 12.9739905, 12.9606321, 12.9962979, 12.9277245, 12.9986827, 12.9755264, 12.97364413065367, 12.972228669472916, 12.975700066194872, 12.9778793, 12.96668780296818, 12.986391, 12.980005949674641, 12.975801449285402, 12.985725778251844, 12.983009663675498, 12.982242260731894, 12.9934283, 12.9624669, 12.9408685, 12.9467081, 12.9678074, 12.982970719351052, 12.9931876, 13.0093455, 12.9390255, 12.9779079, 12.957998, 12.97339325, 12.9578658, 12.96381425, 12.9876393, 12.9592202, 12.9243692, 12.9282918, 12.9327778, 12.947945899082507, 12.958817250351021, 13.0005359, 13.007516, 13.0227204, 13

In [12]:
lat_long['lat'] = lat
lat_long['lon'] = lon
lat_long.isnull().sum()

Name    0
lat     2
lon     2
dtype: int64

In [13]:
lat_long[lat_long['lat'].isnull()]

,Name,lat,lon
74,"Rammurthy Nagar, Bangalore, Karnataka, India",NaN,NaN
82,"Sadashiv Nagar, Bangalore, Karnataka, India",NaN,NaN


In [40]:
df_temp = zomato2['location'].value_counts().reset_index()
print(df_temp.head(6))
print(df_temp.info())

                                            location  count
0                   BTM, Bangalore, Karnataka, India   4234
1  Koramangala 5th Block, Bangalore, Karnataka, I...   2381
2                   HSR, Bangalore, Karnataka, India   2117
3           Indiranagar, Bangalore, Karnataka, India   1933
4              JP Nagar, Bangalore, Karnataka, India   1842
5             Jayanagar, Bangalore, Karnataka, India   1716
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 92 entries, 0 to 91
Data columns (total 2 columns):
 #   Column    Non-Null Count  Dtype 
---  ------    --------------  ----- 
 0   location  92 non-null     object
 1   count     92 non-null     int64 
dtypes: int64(1), object(1)
memory usage: 1.6+ KB
None


In [41]:
df_temp = df_temp.rename(columns={'location':'Name'})
zomato_stats = df_temp.merge(lat_long, on='Name')
print(zomato_stats.head(3))
print(zomato_stats.tail(3))
print(zomato_stats.info())

                                                Name  count        lat  \
0                   BTM, Bangalore, Karnataka, India   4234  12.916360   
1  Koramangala 5th Block, Bangalore, Karnataka, I...   2381  12.934843   
2                   HSR, Bangalore, Karnataka, India   2117  12.900563   

         lon  
0  77.604733  
1  77.618977  
2  77.649475  
                                                 Name  count        lat  \
89  Rajarajeshwari Nagar, Bangalore, Karnataka, India      2  12.927441   
90            Nagarbhavi, Bangalore, Karnataka, India      1  12.981957   
91                Peenya, Bangalore, Karnataka, India      1  13.033051   

          lon  
89  77.515522  
90  77.518374  
91  77.533223  
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 92 entries, 0 to 91
Data columns (total 4 columns):
 #   Column  Non-Null Count  Dtype  
---  ------  --------------  -----  
 0   Name    92 non-null     object 
 1   count   92 non-null     int64  
 2   lat     90 non-null    

In [42]:
zomato_stats[zomato_stats['lat'].isnull()]

,Name,count,lat,lon
68,"Sadashiv Nagar, Bangalore, Karnataka, India",58,NaN,NaN
73,"Rammurthy Nagar, Bangalore, Karnataka, India",28,NaN,NaN


In [43]:
zomato_stats.loc[73, 'lat'] = 13.0120218
zomato_stats.loc[73, 'lon'] = 77.6777817

zomato_stats.loc[68, 'lat'] = 13.010316
zomato_stats.loc[68,'lon'] = 77.580569
zomato_stats[zomato_stats['lat'].isnull()]

,Name,count,lat,lon


# Visualização
 //visualization

In [50]:
#criando o basemap para a cidade de Bangalore (coords: lat 12.97, lon 77.59)

def create_base_map():
    basemap = folium.Map(location=[12.97, 77.59])
    return basemap

base_map = create_base_map()
base_map

In [51]:
#HeatMap

HeatMap(zomato_stats[['lat', 'lon',
                                    'count']]).add_to(base_map)
base_map

In [52]:
base_map2 = create_base_map()

clustermap = FastMarkerCluster(zomato_stats[['lat',
                                             'lon', 'count']]).add_to(base_map2)
base_map2